In [1]:
import pandas as pd

# تحديث المسار للمكان الفعلي للملف
us_data_path = r'C:\Users\Fares ali\Documents\Final_Project\API_KEY\US_Accidents_March23.csv'

try:
    print("⏳ جاري سحب عينة (1000 سطر) من الداتا الأمريكية العملاقة...")
    
    # استخدام nrows=1000 عشان الميموري متضربش
    df_us_sample = pd.read_csv(us_data_path, nrows=1000)
    
    print(f"✅ تم سحب العينة بنجاح! عدد الأعمدة المتاحة: {len(df_us_sample.columns)}")
    print("\n📜 قائمة بكل الأعمدة الموجودة في الملف:")
    
    # طباعة كل الأعمدة
    for col in df_us_sample.columns:
        print(f"- {col}")
        
    print("\n👀 نظرة على أول سطرين:")
    display(df_us_sample.head(2))

except Exception as e:
    print(f"❌ حصل خطأ أثناء قراءة الملف: {e}")

⏳ جاري سحب عينة (1000 سطر) من الداتا الأمريكية العملاقة...
✅ تم سحب العينة بنجاح! عدد الأعمدة المتاحة: 46

📜 قائمة بكل الأعمدة الموجودة في الملف:
- ID
- Source
- Severity
- Start_Time
- End_Time
- Start_Lat
- Start_Lng
- End_Lat
- End_Lng
- Distance(mi)
- Description
- Street
- City
- County
- State
- Zipcode
- Country
- Timezone
- Airport_Code
- Weather_Timestamp
- Temperature(F)
- Wind_Chill(F)
- Humidity(%)
- Pressure(in)
- Visibility(mi)
- Wind_Direction
- Wind_Speed(mph)
- Precipitation(in)
- Weather_Condition
- Amenity
- Bump
- Crossing
- Give_Way
- Junction
- No_Exit
- Railway
- Roundabout
- Station
- Stop
- Traffic_Calming
- Traffic_Signal
- Turning_Loop
- Sunrise_Sunset
- Civil_Twilight
- Nautical_Twilight
- Astronomical_Twilight

👀 نظرة على أول سطرين:


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day


In [2]:
import pandas as pd
import numpy as np

# المسار الخاص بالملف
us_data_path = r'C:\Users\Fares ali\Documents\Final_Project\API_KEY\US_Accidents_March23.csv'

# قائمة الـ 15 عموداً المحددة
columns_to_keep = [
    'Severity', 'Start_Time', 'Sunrise_Sunset',
    'Temperature(F)', 'Humidity(%)', 'Wind_Speed(mph)', 'Visibility(mi)', 'Weather_Condition',
    'Crossing', 'Junction', 'Traffic_Signal', 'Bump', 'Roundabout', 'Stop'
]

print("⏳ جاري قراءة الأعمدة المحددة فقط وتوفير مساحة الذاكرة (RAM)...")

try:
    # 1. سحب الأعمدة المحددة
    df_ml = pd.read_csv(us_data_path, usecols=columns_to_keep)
    print(f"✅ تم سحب البيانات بنجاح! إجمالي السجلات: {len(df_ml):,}")

    print("⏳ جاري التنظيف والتحويل الهندسي للخصائص...")

    # 2. التخلص من السجلات الناقصة في الأعمدة الأساسية
    df_ml = df_ml.dropna(subset=['Temperature(F)', 'Humidity(%)', 'Weather_Condition', 'Sunrise_Sunset', 'Visibility(mi)'])

    # 3. تحويل الوحدات القياسية
    df_ml['Temperature_C'] = np.round((df_ml['Temperature(F)'] - 32) * 5.0 / 9.0, 1)
    df_ml['Wind_Speed_ms'] = np.round(df_ml['Wind_Speed(mph)'] * 0.44704, 1)
    df_ml['Visibility_km'] = np.round(df_ml['Visibility(mi)'] * 1.60934, 1)

    # 4. استخراج الخصائص الزمنية
    df_ml['Start_Time'] = pd.to_datetime(df_ml['Start_Time'], errors='coerce')
    df_ml['Hour'] = df_ml['Start_Time'].dt.hour
    df_ml['DayOfWeek'] = df_ml['Start_Time'].dt.dayofweek  # 0=Monday, 6=Sunday

    # 5. تحويل المتغيرات المنطقية (True/False) إلى (1/0)
    bool_cols = ['Crossing', 'Junction', 'Traffic_Signal', 'Bump', 'Roundabout', 'Stop']
    for col in bool_cols:
        df_ml[col] = df_ml[col].astype(int)

    # 6. حذف الأعمدة الأصلية التي تم استبدالها
    df_ml = df_ml.drop(columns=['Temperature(F)', 'Wind_Speed(mph)', 'Visibility(mi)', 'Start_Time'])

    print(f"🚀 اكتملت عملية التجهيز! حجم البيانات النهائي: {len(df_ml):,} سجل.")
    display(df_ml.head())

except Exception as e:
    print(f"❌ حدث خطأ: {e}")

⏳ جاري قراءة الأعمدة المحددة فقط وتوفير مساحة الذاكرة (RAM)...
✅ تم سحب البيانات بنجاح! إجمالي السجلات: 7,728,394
⏳ جاري التنظيف والتحويل الهندسي للخصائص...
🚀 اكتملت عملية التجهيز! حجم البيانات النهائي: 7,478,390 سجل.


,Severity,Humidity(%),Weather_Condition,Bump,Crossing,Junction,Roundabout,Stop,Traffic_Signal,Sunrise_Sunset,Temperature_C,Wind_Speed_ms,Visibility_km,Hour,DayOfWeek
0,3,91.0,Light Rain,0,0,0,0,0,0,Night,2.7,NaN,16.1,5.0,0.0
1,2,100.0,Light Rain,0,0,0,0,0,0,Night,3.3,NaN,16.1,6.0,0.0
2,2,100.0,Overcast,0,0,0,0,0,1,Night,2.2,1.6,16.1,6.0,0.0
3,3,96.0,Mostly Cloudy,0,0,0,0,0,0,Night,1.7,2.1,14.5,7.0,0.0
4,2,89.0,Mostly Cloudy,0,0,0,0,0,1,Day,2.2,1.6,9.7,7.0,0.0


In [7]:
#%pip install scikit-learn

In [13]:
from sklearn.preprocessing import LabelEncoder

print("⏳ جاري معالجة القيم المفقودة المتبقية...")
# 1. تعويض الـ NaN في سرعة الرياح بمتوسط السرعة
median_wind = df_ml['Wind_Speed_ms'].median()
df_ml['Wind_Speed_ms'] = df_ml['Wind_Speed_ms'].fillna(median_wind)

print("⏳ جاري تحويل النصوص إلى أرقام (Encoding)...")
# 2. تحويل الليل والنهار إلى 0 و 1
df_ml['Sunrise_Sunset'] = df_ml['Sunrise_Sunset'].map({'Day': 1, 'Night': 0})
# لو في أي NaN متبقي بالصدفة هنا، نخليه 1 (نهار) كقيمة افتراضية
df_ml['Sunrise_Sunset'] = df_ml['Sunrise_Sunset'].fillna(1).astype(int)

# 3. تحويل حالات الطقس (Weather_Condition) لأرقام
le = LabelEncoder()
df_ml['Weather_Condition'] = le.fit_transform(df_ml['Weather_Condition'].astype(str))

print("⏳ جاري سحب عينة للتدريب لتجنب انهيار الذاكرة (RAM)...")
# 4. أخذ عينة عشوائية (500 ألف سجل) لسرعة التدريب
df_sample = df_ml.sample(n=500000, random_state=42)

# 5. فصل الميزات (X) عن الهدف (y)
X = df_sample.drop(columns=['Severity'])
y = df_sample['Severity']

print("✅ تم التجهيز بنجاح!")
print(f"📊 أبعاد مصفوفة التدريب (X): {X.shape}")
print(f"🎯 أبعاد مصفوفة الهدف (y): {y.shape}")
display(X.head(31))

⏳ جاري معالجة القيم المفقودة المتبقية...
⏳ جاري تحويل النصوص إلى أرقام (Encoding)...
⏳ جاري سحب عينة للتدريب لتجنب انهيار الذاكرة (RAM)...
✅ تم التجهيز بنجاح!
📊 أبعاد مصفوفة التدريب (X): (500000, 14)
🎯 أبعاد مصفوفة الهدف (y): (500000,)


,Humidity(%),Weather_Condition,Bump,Crossing,Junction,Roundabout,Stop,Traffic_Signal,Sunrise_Sunset,Temperature_C,Wind_Speed_ms,Visibility_km,Hour,DayOfWeek
3504750,20.0,142,0,0,0,0,0,0,1,27.8,5.1,16.1,12.0,1.0
6757547,33.0,90,0,0,0,0,0,0,1,12.8,0.0,16.1,0.0,3.0
4177448,39.0,32,0,0,0,0,1,0,1,19.4,0.0,16.1,10.0,0.0
7451101,87.0,36,0,0,0,0,0,0,1,6.0,2.1,16.1,18.0,0.0
4411202,55.0,32,0,0,0,0,0,1,1,30.0,4.5,16.1,15.0,3.0
920851,46.0,40,0,1,0,0,0,1,1,34.4,3.1,16.1,13.0,4.0
1785357,66.0,90,0,0,0,0,0,0,1,7.2,1.3,16.1,18.0,6.0
5079440,70.0,15,0,0,0,0,0,0,1,-13.3,6.3,3.2,NaN,NaN
4258851,48.0,90,0,0,0,0,0,0,1,11.7,4.5,16.1,NaN,NaN
4883838,100.0,15,0,0,1,0,0,0,1,0.0,0.0,1.2,11.0,5.0


In [14]:
# 1. نظرة عامة على أنواع البيانات والذاكرة
print("📊 1. معلومات الجدول (Data Info):")
df_sample.info()

print("\n" + "="*40 + "\n")

# 2. حصر القيم المفقودة (Null Values) في كل عمود
print("❌ 2. القيم المفقودة (Null Values):")
null_counts = df_sample.isnull().sum()
display(null_counts[null_counts > 0]) # هيعرض الأعمدة اللي فيها Null بس عشان الزحمة

print("\n" + "="*40 + "\n")

# 3. فحص الصفوف المكررة بالملي (Duplicates)
duplicates_count = df_sample.duplicated().sum()
print(f"👯 3. عدد الصفوف المكررة (Duplicates): {duplicates_count:,} صف")

# 4. لو في تكرار، نعرض عينة منه عشان نفهم سببه
if duplicates_count > 0:
    print("\n👀 عينة من الصفوف المكررة:")
    display(df_sample[df_sample.duplicated(keep=False)].sort_values(by=list(df_sample.columns)).head(4))

📊 1. معلومات الجدول (Data Info):
<class 'pandas.DataFrame'>
Index: 500000 entries, 3504750 to 4568528
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Severity           500000 non-null  int64  
 1   Humidity(%)        500000 non-null  float64
 2   Weather_Condition  500000 non-null  int64  
 3   Bump               500000 non-null  int64  
 4   Crossing           500000 non-null  int64  
 5   Junction           500000 non-null  int64  
 6   Roundabout         500000 non-null  int64  
 7   Stop               500000 non-null  int64  
 8   Traffic_Signal     500000 non-null  int64  
 9   Sunrise_Sunset     500000 non-null  int64  
 10  Temperature_C      500000 non-null  float64
 11  Wind_Speed_ms      500000 non-null  float64
 12  Visibility_km      500000 non-null  float64
 13  Hour               452437 non-null  float64
 14  DayOfWeek          452437 non-null  float64
dtypes: float64(6), int64(9)
mem

Hour         47563
DayOfWeek    47563
dtype: int64



👯 3. عدد الصفوف المكررة (Duplicates): 20,049 صف

👀 عينة من الصفوف المكررة:


,Severity,Humidity(%),Weather_Condition,Bump,Crossing,Junction,Roundabout,Stop,Traffic_Signal,Sunrise_Sunset,Temperature_C,Wind_Speed_ms,Visibility_km,Hour,DayOfWeek
6965766,1,5.0,90,0,1,0,0,0,1,1,38.9,7.6,16.1,16.0,3.0
6965764,1,5.0,90,0,1,0,0,0,1,1,38.9,7.6,16.1,16.0,3.0
6935383,1,11.0,90,0,1,0,0,0,1,1,35.0,0.0,16.1,12.0,2.0
6935381,1,11.0,90,0,1,0,0,0,1,1,35.0,0.0,16.1,12.0,2.0


In [11]:
print("🧹 ابدأ عملية التنظيف الشاملة...")

# 1. معالجة القيم المفقودة في Hour و DayOfWeek
# هنعوض الـ NaN بقيمة الـ Mode (القيمة الأكثر تكراراً) لكل عمود
mode_hour = df_sample['Hour'].mode()[0]
mode_day = df_sample['DayOfWeek'].mode()[0]

df_sample['Hour'] = df_sample['Hour'].fillna(mode_hour)
df_sample['DayOfWeek'] = df_sample['DayOfWeek'].fillna(mode_day)

# 2. التأكد من تحويل نوع البيانات لأرقام صحيحة بعد تعويض الـ Nulls
df_sample['Hour'] = df_sample['Hour'].astype(int)
df_sample['DayOfWeek'] = df_sample['DayOfWeek'].astype(int)

# 3. حذف الصفوف المكررة (Duplicates) تماماً
initial_rows = len(df_sample)
df_sample = df_sample.drop_duplicates()
removed_rows = initial_rows - len(df_sample)
print(f"🗑️ تم حذف {removed_rows:,} صف مكرر بنجاح.")

# 4. فصل الميزات (X) عن الهدف (y) النهائيين بعد التنظيف
X_clean = df_sample.drop(columns=['Severity'])
y_clean = df_sample['Severity']

print(f"\n✨ الداتا بقت نظيفة وجاهزة تماماً للتدريب!")
print(f"📊 حجم مصفوفة الميزات (X_clean): {X_clean.shape}")
print(f"🎯 حجم مصفوفة الهدف (y_clean): {y_clean.shape}")

🧹 ابدأ عملية التنظيف الشاملة...
🗑️ تم حذف 20,594 صف مكرر بنجاح.

✨ الداتا بقت نظيفة وجاهزة تماماً للتدريب!
📊 حجم مصفوفة الميزات (X_clean): (479406, 14)
🎯 حجم مصفوفة الهدف (y_clean): (479406,)


In [18]:
display(df_sample.head(4))
df_sample.info()


,Severity,Humidity(%),Weather_Condition,Bump,Crossing,Junction,Roundabout,Stop,Traffic_Signal,Sunrise_Sunset,Temperature_C,Wind_Speed_ms,Visibility_km,Hour,DayOfWeek
3504750,4,20.0,142,0,0,0,0,0,0,1,27.8,5.1,16.1,12.0,1.0
6757547,4,33.0,90,0,0,0,0,0,0,1,12.8,0.0,16.1,0.0,3.0
4177448,2,39.0,32,0,0,0,0,1,0,1,19.4,0.0,16.1,10.0,0.0
7451101,4,87.0,36,0,0,0,0,0,0,1,6.0,2.1,16.1,18.0,0.0


<class 'pandas.DataFrame'>
Index: 500000 entries, 3504750 to 4568528
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Severity           500000 non-null  int64  
 1   Humidity(%)        500000 non-null  float64
 2   Weather_Condition  500000 non-null  int64  
 3   Bump               500000 non-null  int64  
 4   Crossing           500000 non-null  int64  
 5   Junction           500000 non-null  int64  
 6   Roundabout         500000 non-null  int64  
 7   Stop               500000 non-null  int64  
 8   Traffic_Signal     500000 non-null  int64  
 9   Sunrise_Sunset     500000 non-null  int64  
 10  Temperature_C      500000 non-null  float64
 11  Wind_Speed_ms      500000 non-null  float64
 12  Visibility_km      500000 non-null  float64
 13  Hour               452437 non-null  float64
 14  DayOfWeek          452437 non-null  float64
dtypes: float64(6), int64(9)
memory usage: 61.0 MB


In [21]:
# 1. إعادة ضبط الإندكس وترتيبه بعد الحذف والتنظيف
df_clean = df_sample.drop_duplicates().reset_index(drop=True)

# 2. عرض أول 3 صفوف (Head)
print("👀 أول 3 صفوف في الداتا النظيفة:")
display(df_clean.head(3))

# 3. عرض آخر 3 صفوف (Tail) للتأكد من النهاية
print("\n👀 آخر 3 صفوف في الداتا النظيفة:")
display(df_clean.tail(3))

# 4. فحص الـ Info والأنواع بعد التنظيف النهائي
print("\n📊 معلومات الجدول بعد التنظيف (Info):")
df_clean.info()

# 5. عرض الإحصائيات السريعة (Describe) للتأكد من عدم وجود قيم شاذة (Outliers)
print("\n📈 ملخص الإحصائيات (Describe):")
display(df_clean.describe())

# 6. تحديث المتغيرات النهائية للتدريب
X_final = df_clean.drop(columns=['Severity'])
y_final = df_clean['Severity']

print(f"\n🚀 جاهزين للخطوة اللي بعدها! الإجمالي النهائي للسجلات: {len(df_clean):,}")

👀 أول 3 صفوف في الداتا النظيفة:


,Severity,Humidity(%),Weather_Condition,Bump,Crossing,Junction,Roundabout,Stop,Traffic_Signal,Sunrise_Sunset,Temperature_C,Wind_Speed_ms,Visibility_km,Hour,DayOfWeek
0,4,20.0,142,0,0,0,0,0,0,1,27.8,5.1,16.1,12.0,1.0
1,4,33.0,90,0,0,0,0,0,0,1,12.8,0.0,16.1,0.0,3.0
2,2,39.0,32,0,0,0,0,1,0,1,19.4,0.0,16.1,10.0,0.0



👀 آخر 3 صفوف في الداتا النظيفة:


,Severity,Humidity(%),Weather_Condition,Bump,Crossing,Junction,Roundabout,Stop,Traffic_Signal,Sunrise_Sunset,Temperature_C,Wind_Speed_ms,Visibility_km,Hour,DayOfWeek
479948,2,40.0,90,0,0,0,0,0,0,1,31.1,7.6,16.1,12.0,4.0
479949,2,65.0,90,0,1,0,0,0,1,1,27.2,0.0,16.1,10.0,0.0
479950,2,58.0,90,0,0,0,0,0,0,1,15.0,4.0,16.1,NaN,NaN



📊 معلومات الجدول بعد التنظيف (Info):
<class 'pandas.DataFrame'>
RangeIndex: 479951 entries, 0 to 479950
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Severity           479951 non-null  int64  
 1   Humidity(%)        479951 non-null  float64
 2   Weather_Condition  479951 non-null  int64  
 3   Bump               479951 non-null  int64  
 4   Crossing           479951 non-null  int64  
 5   Junction           479951 non-null  int64  
 6   Roundabout         479951 non-null  int64  
 7   Stop               479951 non-null  int64  
 8   Traffic_Signal     479951 non-null  int64  
 9   Sunrise_Sunset     479951 non-null  int64  
 10  Temperature_C      479951 non-null  float64
 11  Wind_Speed_ms      479951 non-null  float64
 12  Visibility_km      479951 non-null  float64
 13  Hour               438735 non-null  float64
 14  DayOfWeek          438735 non-null  float64
dtypes: float64(6), int64(9)


,Severity,Humidity(%),Weather_Condition,Bump,Crossing,Junction,Roundabout,Stop,Traffic_Signal,Sunrise_Sunset,Temperature_C,Wind_Speed_ms,Visibility_km,Hour,DayOfWeek
count,479951.000000,479951.000000,479951.000000,479951.000000,479951.000000,479951.000000,479951.000000,479951.000000,479951.000000,479951.0,479951.000000,479951.000000,479951.000000,438735.000000,438735.000000
mean,2.218729,64.850641,66.575119,0.000448,0.116827,0.076593,0.000031,0.028507,0.153499,1.0,16.453323,3.436815,14.619791,12.270494,2.562738
std,0.491298,22.859242,42.218335,0.021160,0.321214,0.265945,0.005590,0.166417,0.360468,0.0,10.563380,2.310388,4.368138,5.441091,1.795026
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,-32.200000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,48.000000,32.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,9.400000,2.200000,16.100000,8.000000,1.000000
50%,2.000000,67.000000,90.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,17.800000,3.100000,16.100000,13.000000,3.000000
75%,2.000000,84.000000,90.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,24.400000,4.500000,16.100000,17.000000,4.000000
max,4.000000,100.000000,142.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,91.100000,114.000000,178.600000,23.000000,6.000000



🚀 جاهزين للخطوة اللي بعدها! الإجمالي النهائي للسجلات: 479,951


In [23]:
# 1. تعويض أي NaN متبقي في الـ Hour والـ DayOfWeek بقيمة الـ Mode
df_clean['Hour'] = df_clean['Hour'].fillna(df_clean['Hour'].mode()[0]).astype(int)
df_clean['DayOfWeek'] = df_clean['DayOfWeek'].fillna(df_clean['DayOfWeek'].mode()[0]).astype(int)

# 2. التأكيد النهائي على نظافة الداتا
print("✨ عدد الـ Nulls الحالي في كل الأعمدة:")
print(df_clean.isnull().sum())
df_clean.info()

✨ عدد الـ Nulls الحالي في كل الأعمدة:
Severity             0
Humidity(%)          0
Weather_Condition    0
Bump                 0
Crossing             0
Junction             0
Roundabout           0
Stop                 0
Traffic_Signal       0
Sunrise_Sunset       0
Temperature_C        0
Wind_Speed_ms        0
Visibility_km        0
Hour                 0
DayOfWeek            0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 479951 entries, 0 to 479950
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Severity           479951 non-null  int64  
 1   Humidity(%)        479951 non-null  float64
 2   Weather_Condition  479951 non-null  int64  
 3   Bump               479951 non-null  int64  
 4   Crossing           479951 non-null  int64  
 5   Junction           479951 non-null  int64  
 6   Roundabout         479951 non-null  int64  
 7   Stop               479951 non-null  int64  
 8   Traffic_S

In [24]:
# تحديد اسم الملف الجديد
clean_output_file = 'Cleaned_US_Accidents_Sample.csv'

# حفظ البيانات النظيفة مع منع حفظ الإندكس الزيادة
df_clean.to_csv(clean_output_file, index=False, encoding='utf-8-sig')

print(f"💾 تم حفظ الداتا النظيفة بنجاح!")
print(f"📁 اسم الملف: {clean_output_file}")
print(f"📊 عدد السجلات المحفوظة: {len(df_clean):,}")

💾 تم حفظ الداتا النظيفة بنجاح!
📁 اسم الملف: Cleaned_US_Accidents_Sample.csv
📊 عدد السجلات المحفوظة: 479,951
